In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import gzip

warnings.filterwarnings("ignore")

In [2]:
TRAIN_PATH = "/kaggle/input/competitions/avazu-ctr-prediction/train.gz"
SAMPLE_SIZE = 1000000
RANDOM_SEED = 42

In [3]:
COL_NOTES = {
    "id":           "Ad identifier (unique per row)",
    "click":        "Target: 1 = clicked, 0 = not clicked",
    "hour":         "YYMMDDHH format — encodes date AND hour",
    "C1":           "Anonymized categorical",
    "banner_pos":   "Position of the banner on the page",
    "site_id":      "Site identifier",
    "site_domain":  "Domain the site belongs to",
    "site_category":"Category of the site",
    "app_id":       "App identifier",
    "app_domain":   "Domain of the app",
    "app_category": "Category of the app",
    "device_id":    "Device identifier (very high cardinality)",
    "device_ip":    "IP address (very high cardinality)",
    "device_model": "Device model",
    "device_type":  "Type of device (phone/tablet/…)",
    "device_conn_type": "Connection type (wifi/3g/…)",
    "C14–C21":      "Anonymized categoricals",
}

for k,v in COL_NOTES.items():
    print(f"{k:20s}:   {v}")

id                  :   Ad identifier (unique per row)
click               :   Target: 1 = clicked, 0 = not clicked
hour                :   YYMMDDHH format — encodes date AND hour
C1                  :   Anonymized categorical
banner_pos          :   Position of the banner on the page
site_id             :   Site identifier
site_domain         :   Domain the site belongs to
site_category       :   Category of the site
app_id              :   App identifier
app_domain          :   Domain of the app
app_category        :   Category of the app
device_id           :   Device identifier (very high cardinality)
device_ip           :   IP address (very high cardinality)
device_model        :   Device model
device_type         :   Type of device (phone/tablet/…)
device_conn_type    :   Connection type (wifi/3g/…)
C14–C21             :   Anonymized categoricals


### Step 1: Loading Sample

In [4]:
with gzip.open(TRAIN_PATH,'rb') as f:
    total_rows = sum(1 for _ in f)

print("Total number of rows:", total_rows)

Total number of rows: 40428968


In [5]:
skip_prob = 1-(SAMPLE_SIZE/total_rows)
skip_rows = np.random.RandomState(RANDOM_SEED).choice(
    range(1,total_rows+1),
    size = (total_rows-SAMPLE_SIZE),
    replace=False
)
print(skip_prob)

0.9752652602955386


In [6]:
df = pd.read_csv(TRAIN_PATH,skiprows=skip_rows)
print(f"Loaded sample of rows {df.shape[0]} and columns {df.shape[1]}")
df.head()

Loaded sample of rows 1000000 and columns 24


,id,click,hour,C1,banner_pos,site_id,site_domain,site_category,app_id,app_domain,...,device_type,device_conn_type,C14,C15,C16,C17,C18,C19,C20,C21
0,10000679056417042096,0,14102100,1005,1,fe8cc448,9166c161,0569f928,ecad2386,7801e8d9,...,1,0,18993,320,50,2161,0,35,-1,157
1,10011205200760015892,0,14102100,1005,0,6256f5b4,28f93029,f028772b,ecad2386,7801e8d9,...,1,0,17212,320,50,1887,3,39,100202,23
2,10017756797097747189,0,14102100,1005,0,543a539e,c7ca3108,3e814130,ecad2386,7801e8d9,...,1,0,20366,320,50,2333,0,39,-1,157
3,10027325869092401811,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,...,1,0,15702,320,50,1722,0,35,100084,79
4,10028659945951062490,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,...,1,0,15706,320,50,1722,0,35,100084,79


### Step 2: Checks

In [7]:
df.dtypes

id                  uint64
click                int64
hour                 int64
C1                   int64
banner_pos           int64
site_id             object
site_domain         object
site_category       object
app_id              object
app_domain          object
app_category        object
device_id           object
device_ip           object
device_model        object
device_type          int64
device_conn_type     int64
C14                  int64
C15                  int64
C16                  int64
C17                  int64
C18                  int64
C19                  int64
C20                  int64
C21                  int64
dtype: object

In [8]:
df.isnull().sum()

id                  0
click               0
hour                0
C1                  0
banner_pos          0
site_id             0
site_domain         0
site_category       0
app_id              0
app_domain          0
app_category        0
device_id           0
device_ip           0
device_model        0
device_type         0
device_conn_type    0
C14                 0
C15                 0
C16                 0
C17                 0
C18                 0
C19                 0
C20                 0
C21                 0
dtype: int64

In [9]:
click_rate = df['click'].mean()
click_rate

np.float64(0.169756)

In [10]:
print(f"Clicks {df['click'].sum()}")
print(f"Non clicks {(df['click']==0).sum()}")
print(f"\n Imbalance ratio: 1:{(1-click_rate)/click_rate:.1f}  ")

Clicks 169756
Non clicks 830244

 Imbalance ratio: 1:4.9  
